In [1]:
!pip install earthengine-api geemap pillow imageio -q
#Important---------Hayli Gubbi Volcano
import ee
import geemap
from PIL import Image, ImageDraw
import requests
import numpy as np
import imageio

# -------------------------------
# Initialize EE
# -------------------------------
ee.Authenticate()
ee.Initialize(project='satelliteanalysis')

# -------------------------------
# AOI and Volcano Info
# -------------------------------
volcano_name = "Hayli Gubbi Volcano"
lat, lon = 13.5111, 40.7161
hayli = ee.Geometry.Polygon([
  [
    [
      40.67226919523229,
      13.471849105631842
    ],
    [
      40.76908621183385,
      13.471849105631842
    ],
    [
      40.76908621183385,
      13.557306073819241
    ],
    [
      40.67226919523229,
      13.557306073819241
    ],
    [
      40.67226919523229,
      13.471849105631842
    ]
  ]
])

# Zoomed-out buffer: 3000 km
aoi = hayli#.buffer(3000)  #3000 km buffer

# -------------------------------
# Strict cloud mask
# -------------------------------
def mask_s2_sr_strict(image):
    qa = image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    scl = image.select('SCL')
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(cloud_mask).updateMask(scl_mask)

# -------------------------------
# Load Sentinel-2 SR collection (all passovers)
# -------------------------------
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(aoi)
      .filterDate("2025-10-10", "2025-12-21")
      #.map(mask_s2_sr_strict)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
      .sort('system:time_start'))  # Sort by acquisition date

# -------------------------------
# Convert to list for iteration
# -------------------------------
images = s2.toList(s2.size())
raw_frames = []

for i in range(images.size().getInfo()):
    img = ee.Image(images.get(i))

    # Clip to AOI
    img = img.clip(aoi).unmask(0)

    # -------------------------------
    # True-color RGB composite
    # B4: Red, B3: Green, B2: Blue
    # -------------------------------
    rgb = img.select(['B4','B3','B2'])
    #rgb = img.select(['B8','B4','B3'])
    #vis = rgb.visualize(min=217, max=3000)
    vis = rgb.visualize(min=0, max=2000)

    # Download image
    url = vis.getThumbURL({'region': aoi, 'dimensions': 512})
    im = Image.open(requests.get(url, stream=True).raw).convert('RGB')

    # Footer with volcano name, coords, date
    draw = ImageDraw.Draw(im)
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    footer_text = f"{volcano_name} | Lat: {lat:.4f}, Lon: {lon:.4f} | {date_str}"
    rect_height = 35
    draw.rectangle([(0, im.height - rect_height), (im.width, im.height)], fill=(0,0,0,180))
    draw.text((10, im.height - rect_height + 5), footer_text, fill=(255,255,255))

    # Resize
    im = im.resize((1920,1080))
    raw_frames.append(np.array(im))

# -------------------------------
# Smooth transitions between frames
# -------------------------------
smooth_frames = []
for i in range(len(raw_frames)-1):
    f1 = raw_frames[i].astype(np.float32)
    f2 = raw_frames[i+1].astype(np.float32)
    smooth_frames.append(f1.astype(np.uint8))
    for alpha in [0.25,0.5,0.75]:
        interp = (f1*(1-alpha)+f2*alpha).astype(np.uint8)
        smooth_frames.append(interp)
smooth_frames.append(raw_frames[-1])

# -------------------------------
# Save  timelapse video
# -------------------------------
video_file = "HayliGubbi_RGB_AllPassovers.mp4"
imageio.mimwrite(video_file, smooth_frames, fps=2)
print(f"Video saved: {video_file}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.4 MB/s eta 0:00:00


Video saved: HayliGubbi_RGB_AllPassovers.mp4
